## 04 - XDec Deconvolution (complete pipeline: Stage 0 -> k-sweep -> Stage 1 -> Stage 2)

### Aim
* Run XDec (`BRL-BCM/XDec_SM`) end-to-end on the full 182-sample TCGA-PAAD cohort: reference export -> Stage 0 gene selection -> k-stability sweep -> Stage 1 (proportions) -> Stage 2 (per-cell-type expression). Produces XDec's independent, reference-anchored deconvolution estimate for cross-tool comparison against CIBERSORTx (05) and TIMER (06); that comparison itself, plus all downstream biological interpretation (cell-type labeling, DE/enrichment), lives in the EDA notebook (07).
* **Reference:** `Zenodo 3969339` (Peng et al. 2019, Besca-reprocessed) - chosen over ~12 other candidate PDAC single-cell sources for full whole-tumor cell-type coverage, PDAC specificity, size (57,423 cells/35 patients), and being the exact source behind the PMID 36274954 benchmark paper; small enough (1.68 GB) to load locally. Full candidate comparison archived in `archive/04_deconvolution_plan.ipynb`.
* **Kernel note:** Python 3, not this project's usual `ir` (R) kernel - XDec is a Python CLI tool, so this avoids an R-shells-out-to-Python indirection.

### Input
* `data/processed/TCGA-PAAD_counts_raw.tsv` (raw counts, notebook 02) - XDec's bulk input.
* `data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad` - Peng et al.'s reprocessed single-cell reference.

### Output
* `data/deconvolution/xdec/refs.tsv` / `ref_metadata.tsv` - exported reference (see "Technical wrinkle" note below for the `expm1()` fix needed).
* `data/deconvolution/xdec/stage0/selected_probes.txt` - 344-gene one-vs-rest informative set.
* `data/deconvolution/xdec/full_sweep/{stability_scores.tsv, optimal_k.txt, timing_log.tsv}` - `k=3..12` stability sweep.
* `data/deconvolution/xdec/stage1/{proportions.tsv, cell_type_profiles.tsv}` - main pipeline, no refs, k=optimal.
* `data/deconvolution/xdec/stage1_with_refs/{proportions.tsv, cell_type_profiles.tsv, correlation.tsv}` and `..._k8/` - refs-labeled runs at k=3 (optimal) and k=8 (near-tied stability candidate, see sweep result below).
* `data/deconvolution/xdec/stage2/`, `stage2_with_refs_k3/`, `stage2_with_refs_k8/` - matching Stage 2 (NNLS expression) output for each.

### Steps
1. Export Peng et al.'s reference to `--refs`/`--ref-metadata` TSVs.
2. `xdec.py stage0 --skip-stability` for gene selection only.
3. Full `k=3..12` stability sweep, partitioned into 10 single-k subprocesses (3 concurrent) so an hours-long unattended run survives interruption - see rationale below.
4. Stage 1 + Stage 2, no refs, at the chosen k (main pipeline).
5. Stage 1 + Stage 2, with refs (for correlation-based cell-type labeling), at k=3 and k=8.

All algorithm parameters (tests, convergence, randomness, stability-test repetitions) are XDec's unmodified defaults; only execution flow (partitioning/concurrency) was changed.

### Result
* All 5 steps completed. `optimal_k = 3` (stability 1.0000), with k=8 essentially tied (0.9995) - both carried forward since k=8 turned out to resolve more biologically distinct profiles (see notebook 07).

### Conclusion
* Self-contained, idempotent, resumable record of the full XDec pipeline. Output used in the cross-tool comparison and biological validation in notebook 07.

## Reference choice: Peng et al. 2019, reprocessed (`Zenodo 3969339`)

~12 candidate PDAC single-cell/reference sources were checked (GEO, Zenodo, SND, HTAN, CELLxGENE Census); most failed on cell-type coverage (fibroblast-only, immune-only), unverifiable sorting, mouse-only data, restricted access, or being too large for local RAM (1M+ cells). Full candidate-by-candidate table archived in `archive/04_deconvolution_plan.ipynb`.

**Chosen: `Zenodo 3969339`** - Peng et al. 2019 (`CRA001160`), Besca-reprocessed. Full whole-tumor coverage (ductal, fibroblast, pancreatic stellate, endothelial, acinar, endocrine, detailed immune subsets), PDAC-specific, 57,423 cells / 35 patients, and the exact source data behind the PMID 36274954 benchmark paper. Verified on download (1,681,192,568 bytes, exact match), small enough to load locally (1.68 GB).

**Technical wrinkle, handled in the export step below:** neither matrix in the annotated h5ad is raw counts. `adata.X` is z-scored (negative values, unusable). `adata.raw.X` is non-negative but **natural-log**-normalized (`ln(x+1)`), not `log2(x+1)` - so it needs `expm1()` (not XDec's `--log2-refs`, which inverts `log2(x+1)` specifically) before being written to the `--refs` TSV, then run **without** `--log2-refs`.

### Set working directory to the project root

In [1]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "rnaseq_download":
    os.chdir(os.path.join("..", ".."))
os.getcwd()

'/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress'

### Step 1: export Peng et al.'s reference (with the expm1 fix), inlined natively

In [2]:
xdec_repo = "tools/XDec_SM"
xdec_script = os.path.join(xdec_repo, "python", "xdec.py")
if not os.path.exists(xdec_script):
    os.makedirs(os.path.dirname(xdec_repo) or ".", exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/BRL-BCM/XDec_SM.git", xdec_repo], check=True)
assert os.path.exists(xdec_script)

xdec_dir = "data/deconvolution/xdec"
os.makedirs(xdec_dir, exist_ok=True)
refs_path = os.path.join(xdec_dir, "refs.tsv")
meta_path = os.path.join(xdec_dir, "ref_metadata.tsv")

if not os.path.exists(refs_path):
    import scanpy as sc

    adata = sc.read_h5ad("data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad")
    class_col = "celltype1"
    max_cells_per_class = 300
    rng = np.random.default_rng(1)

    classes = adata.obs[class_col].astype(str)
    keep_names = []
    for cls, names in classes.groupby(classes).groups.items():
        names = np.array(names)  # cell barcodes (obs_names), not positional indices
        n = min(max_cells_per_class, len(names))
        keep_names.extend(rng.choice(names, size=n, replace=False))

    sub_raw = adata.raw[keep_names]
    counts_like = np.expm1(sub_raw.X)
    try:
        counts_like = counts_like.toarray()
    except AttributeError:
        pass
    counts_like = np.asarray(counts_like)
    counts_like[counts_like < 0] = 0.0  # guard against floating-point noise below zero

    cells = pd.Index(keep_names)
    refs_df = pd.DataFrame(counts_like.T, index=sub_raw.var_names, columns=cells)
    refs_df.index.name = "Genes"
    refs_df.to_csv(refs_path, sep="\t")

    meta_df = pd.DataFrame({"Sample_ID": cells, "Cell_Type": classes.loc[cells].values})
    meta_df.to_csv(meta_path, sep="\t", index=False)

    print(f"Wrote {refs_df.shape[0]} genes x {refs_df.shape[1]} cells -> {refs_path}")
    print(meta_df["Cell_Type"].value_counts().to_string())
else:
    print("refs.tsv already exported - skipping.")

refs.tsv already exported - skipping.


### Step 2: Stage 0 gene selection (`--skip-stability` - fast, probe selection only)

In [3]:
import time

stage0_dir = os.path.join(xdec_dir, "stage0")
os.makedirs(stage0_dir, exist_ok=True)
probes_path = os.path.join(stage0_dir, "selected_probes.txt")

if not os.path.exists(probes_path):
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage0",
         "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--refs", refs_path,
         "--ref-metadata", meta_path,
         "--skip-stability",
         "--outdir", stage0_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print(f"Stage 0 (gene selection only) elapsed: {time.time() - t0:.1f} sec")

with open(probes_path) as fh:
    probes = [line.strip() for line in fh if line.strip()]
len(probes)

344

### Step 3: full `k=3..12` stability sweep - 10 partitions, one per individual k value, 3 running concurrently at any time

To be explicit about two separate facts that "partitioned by k, run 3 at a time in parallel" can blur together: (a) **the partition granularity** - there are 10 partitions total, one per individual k value (k=3, k=4, ..., k=12 - each its own subprocess, not k divided into a few chunks/ranges), and (b) **the concurrency level** - 3 of those 10 partitions run at once, the rest queue behind them. These are independent choices: the partition count (10) is fixed by the k-range itself, while the concurrency (3) is a separate hardware-driven choice (see the design note below) and could be set to 1 (fully sequential), 10 (fully parallel), or anything in between without changing what a "partition" means.

**Why partitioned at all:** this run can take hours. `cmd_stage0()` only writes `stability_scores.tsv`/`optimal_k.txt` once, after the entire `k_range` loop finishes - no incremental save inside a single combined call. Any non-clean interruption before the full range completes means zero usable output. Partitioning into one subprocess per k, each writing its own `k_XX/stability_scores.tsv` immediately, means only the currently running k's are at risk.

**Why 3 concurrent workers:** this machine is an Apple M3, 8 cores (4 performance + 4 efficiency), 16GB RAM. Each k's subprocess uses ~130% CPU (~1.3 cores) based on earlier observation - 3 concurrent workers (~4 cores) leaves headroom for the OS/Jupyter/everything else during an hours-long unattended run, rather than saturating all 8 cores and risking thermal throttling on a sustained multi-hour load.

**Documented side effect of partitioning (unchanged by concurrency level - verified from source):** `estimate_stability()` creates `rng = np.random.default_rng(seed)` once per call, before its k loop. Every single-k partition (whether run sequentially or concurrently with others) resets to the same `--seed` (default `12345`) and starts consuming the random stream from the beginning - so every partition draws the same first segment of random sample-subsets, rather than the segment it would get at its true position in one combined `--k-range 3,4,...,12` call. Stability scores from this sweep will be close to, but not bit-identical with, a single combined run. How many partitions run concurrently (1, 3, or 10) does not change this effect at all - it's a property of partitioning by individual k, not of execution order or concurrency.

#### k-sweep: execution strategy and result

**Final per-k result** (`Zenodo 3969339` reference, 344-gene list, 3 concurrent workers, full `k=3..12`):

| k | Elapsed | Stability score |
|---|---|---|
| 3 | 971.4 sec (~16.2 min) | 1.0000 |
| 4 | 1877.9 sec (~31.3 min) | 0.9239 |
| 5 | 2871.7 sec (~47.9 min) | 0.9680 |
| 6 | 5068.6 sec (~84.5 min) | 0.9580 |
| 7 | 7587.5 sec (~126.5 min) | 0.9507 |
| 8 | 11484.8 sec (~191.4 min) | 0.9995 |
| 9 | 7544.9 sec (~125.7 min) | 0.9909 |
| 10 | 13793.9 sec (~229.9 min) | 0.9870 |
| 11 | 16240.9 sec (~270.7 min) | 0.8848 |
| 12 | 14954.4 sec (~249.2 min) | 0.8774 |

Total wall-clock: ~11.3 hours (overnight). **`optimal_k = 3`** by strict argmax; **k=8 essentially tied** (0.9995) - both are carried forward to Stage 1/2 below, since k=8 later turned out to resolve more biologically distinct profiles (notebook 07).

**Why partitioned into 10 single-k subprocesses (3 concurrent), not one combined call:** `cmd_stage0()` only writes results once the *entire* k-range loop finishes, so a single multi-hour call risks losing everything to one interruption - which happened twice during this actual run (one confirmed macOS idle-sleep kill, one unresolved kill despite `caffeinate -i`, fixed by relaunching via `nohup ... & disown`). Partitioning plus per-k checkpointing is why no completed k was lost to either interruption. 3 concurrent workers (not more) was sized to this machine's 8 cores (~1-1.3 cores per worker observed), leaving headroom given the run's eventual multi-hour, all-k duration.

In [4]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

sweep_dir = os.path.join(xdec_dir, "full_sweep")
os.makedirs(sweep_dir, exist_ok=True)
timing_log_path = os.path.join(sweep_dir, "timing_log.tsv")
timing_log_lock = threading.Lock()
if not os.path.exists(timing_log_path):
    with open(timing_log_path, "w") as fh:
        fh.write("k\telapsed_sec\tstability_score\n")


def run_one_k(k):
    k_dir = os.path.join(sweep_dir, f"k_{k:02d}")
    k_stab_tsv = os.path.join(k_dir, "stability_scores.tsv")

    if os.path.exists(k_stab_tsv):
        print(f"k={k} already done - skipping.")
        return

    os.makedirs(k_dir, exist_ok=True)
    print(f"Running k={k} ...")
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage0",
         "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--probes", probes_path,
         "--k-range", str(k),
         "--outdir", k_dir],
        capture_output=True, text=True,
    )
    elapsed_k = time.time() - t0

    if not os.path.exists(k_stab_tsv):
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"k={k} did not produce stability_scores.tsv - stopping here "
            "so completed k's are not lost. See log above."
        )

    stab_k = pd.read_csv(k_stab_tsv, sep="\t")
    score_k = stab_k.loc[stab_k["k"] == k, "stability_score"].iloc[0]

    with timing_log_lock:
        with open(timing_log_path, "a") as fh:
            fh.write(f"{k}\t{elapsed_k:.1f}\t{score_k:.4f}\n")
    print(f"k={k} done: {elapsed_k:.1f} sec, stability={score_k:.4f}")


# 3 workers in parallel - see the markdown cell above for why 3 on this hardware.
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(run_one_k, k): k for k in range(3, 13)}
    for future in as_completed(futures):
        future.result()  # re-raises if run_one_k raised, surfacing the error here

pd.read_csv(timing_log_path, sep="\t")

k=3 already done - skipping.k=4 already done - skipping.

k=5 already done - skipping.
k=6 already done - skipping.
k=7 already done - skipping.
k=8 already done - skipping.
k=9 already done - skipping.
k=10 already done - skipping.
k=11 already done - skipping.
k=12 already done - skipping.


,k,elapsed_sec,stability_score
0,3,971.4,1.0000
1,4,1877.9,0.9239
2,5,2871.7,0.9680
3,6,5068.6,0.9580
4,9,7544.9,0.9909
5,7,7587.5,0.9507
6,8,11484.8,0.9995
7,10,13793.9,0.9870
8,11,16240.9,0.8848
9,12,14954.4,0.8774


### Aggregate all completed k's into the combined stability result

In [5]:
import glob

k_dirs = sorted(glob.glob(os.path.join(sweep_dir, "k_*", "stability_scores.tsv")))
assert len(k_dirs) > 0

all_stab = pd.concat([pd.read_csv(p, sep="\t") for p in k_dirs], ignore_index=True)
all_stab = all_stab.sort_values("k").reset_index(drop=True)

all_stab.to_csv(os.path.join(sweep_dir, "stability_scores.tsv"), sep="\t", index=False)

optimal_k = int(all_stab.loc[all_stab["stability_score"].idxmax(), "k"])
with open(os.path.join(sweep_dir, "optimal_k.txt"), "w") as fh:
    fh.write(str(optimal_k) + "\n")

print(f"Recommended k = {optimal_k} (stability = {all_stab['stability_score'].max():.4f})")
all_stab

Recommended k = 3 (stability = 1.0000)


,k,stability_score
0,3,1.000000
1,4,0.923927
2,5,0.968030
3,6,0.958004
4,7,0.950703
5,8,0.999548
6,9,0.990895
7,10,0.986968
8,11,0.884765
9,12,0.877396


### Step 4: Stage 1 - deconvolve at the chosen k

In [6]:
stage1_dir = os.path.join(xdec_dir, "stage1")
os.makedirs(stage1_dir, exist_ok=True)
proportions_path = os.path.join(stage1_dir, "proportions.tsv")

if not os.path.exists(proportions_path):
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage1",
         "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--probes", probes_path,
         "--k", str(optimal_k),
         "--outdir", stage1_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print(f"Stage 1 elapsed: {time.time() - t0:.1f} sec")

proportions = pd.read_csv(proportions_path, sep="\t", index_col=0)
proportions.head()

,Profile_1,Profile_2,Profile_3
TCGA-IB-7888-01A-11R-2156-07,0.096816,0.623379,0.279805
TCGA-FB-A78T-01A-12R-A32O-07,0.445064,0.403589,0.151347
TCGA-2J-AABI-01A-12R-A41B-07,0.724221,0.082459,0.193320
TCGA-HZ-7918-01A-11R-2156-07,0.099318,0.557294,0.343387
TCGA-2J-AABO-01A-21R-A41B-07,0.096230,0.810188,0.093581


### Step 5: Stage 2 - recover cell-type-specific expression profiles (NNLS, one-shot - not partitioned, since it converges quickly and does not need resumability)

In [7]:
stage2_dir = os.path.join(xdec_dir, "stage2")
os.makedirs(stage2_dir, exist_ok=True)
# Stage 2 actually writes stage2_means.tsv / stage2_stderrs.tsv, not
# cell_type_profiles.tsv (that name is Stage 1's output) - confirmed by
# inspecting the real output after Stage 2 ran; the original guess here was wrong.
means_path = os.path.join(stage2_dir, "stage2_means.tsv")
stderrs_path = os.path.join(stage2_dir, "stage2_stderrs.tsv")

if not os.path.exists(means_path):
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, xdec_script, "stage2",
         "--bulk-counts", "data/processed/TCGA-PAAD_counts_raw.tsv",
         "--proportions", proportions_path,
         "--outdir", stage2_dir],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    print(f"Stage 2 elapsed: {time.time() - t0:.1f} sec")

stage2_means = pd.read_csv(means_path, sep="\t", index_col=0)
stage2_stderrs = pd.read_csv(stderrs_path, sep="\t", index_col=0)
stage2_means.head()

,Profile_1,Profile_2,Profile_3
5S_rRNA,0.477308,0.313488,2.626024
5_8S_rRNA,0.000000,5.417520,0.174918
7SK,0.248836,0.217802,0.000000
A1BG,0.000000,6.171198,30.763223
A1BG-AS1,0.000000,18.284811,140.096641


### Step 6: Stage 1 with refs (correlation-based cell-type labeling), k=3 and k=8

In [8]:
stage1_refs_dirs = {}
for k in (3, 8):
    outdir = os.path.join(xdec_dir, "stage1_with_refs" if k == 3 else f"stage1_with_refs_k{k}")
    os.makedirs(outdir, exist_ok=True)
    prop_path = os.path.join(outdir, "proportions.tsv")
    if not os.path.exists(prop_path):
        t0 = time.time()
        result = subprocess.run(
            [sys.executable, xdec_script, "stage1",
             "--bulk", "data/processed/TCGA-PAAD_counts_raw.tsv",
             "--probes", probes_path,
             "--refs", refs_path,
             "--ref-metadata", meta_path,
             "--k", str(k),
             "--outdir", outdir],
            capture_output=True, text=True,
        )
        print(result.stdout)
        print(result.stderr)
        print(f"Stage 1 (refs, k={k}) elapsed: {time.time() - t0:.1f} sec")
    stage1_refs_dirs[k] = outdir

for k, outdir in stage1_refs_dirs.items():
    shape = pd.read_csv(os.path.join(outdir, "proportions.tsv"), sep="\t", index_col=0).shape
    print(f"k={k}: proportions.tsv shape = {shape}")

k=3: proportions.tsv shape = (182, 3)
k=8: proportions.tsv shape = (182, 8)


### Step 7: Stage 2 with refs, matching k=3 and k=8

In [9]:
for k, outdir in stage1_refs_dirs.items():
    stage2_outdir = os.path.join(xdec_dir, f"stage2_with_refs_k{k}")
    os.makedirs(stage2_outdir, exist_ok=True)
    means_path = os.path.join(stage2_outdir, "stage2_means.tsv")
    if not os.path.exists(means_path):
        t0 = time.time()
        result = subprocess.run(
            [sys.executable, xdec_script, "stage2",
             "--bulk-counts", "data/processed/TCGA-PAAD_counts_raw.tsv",
             "--proportions", os.path.join(outdir, "proportions.tsv"),
             "--outdir", stage2_outdir],
            capture_output=True, text=True,
        )
        print(result.stdout)
        print(result.stderr)
        print(f"Stage 2 (refs, k={k}) elapsed: {time.time() - t0:.1f} sec")
    shape = pd.read_csv(means_path, sep="\t", index_col=0).shape
    print(f"k={k}: stage2_means.tsv shape = {shape}")

k=3: stage2_means.tsv shape = (54603, 3)
k=8: stage2_means.tsv shape = (54603, 8)


## Pipeline scripts & outputs reference (all Stage 0/1/2 runs in this notebook/session)

Includes the two follow-up `--refs`-enabled Stage 1/2 runs (for cell-type labeling, see the next section) alongside the main pipeline's own Stage 0/1/2 steps, since both read from the same `xdec.py` script and the same bulk/reference inputs.

| Stage | Script / command | Key inputs | Output location | Purpose |
|---|---|---|---|---|
| Reference export | inlined Python (Step 1 above), not `xdec.py` itself | `StdWf1_PRJCA001063_annotated.h5ad` | `xdec/refs.tsv`, `xdec/ref_metadata.tsv` | Peng et al. reference, `expm1()`-corrected, subsampled to <=300 cells/class |
| Stage 0 - gene selection | `xdec.py stage0 --refs ... --ref-metadata ... --skip-stability` | `refs.tsv`, `ref_metadata.tsv`, bulk counts | `xdec/stage0/selected_probes.txt` | One-vs-rest t-test -> 344 real informative genes |
| Stage 0 - stability sweep | `xdec.py stage0 --probes ... --k-range <k>` x10, partitioned | `selected_probes.txt`, bulk counts | `xdec/full_sweep/k_XX/stability_scores.tsv` per k, combined `full_sweep/stability_scores.tsv` + `optimal_k.txt` | Pick k via reproducibility across random subsamples |
| Stage 1 - main (no refs, Mode C) | `xdec.py stage1 --probes ... --k 3` | `selected_probes.txt`, bulk counts | `xdec/stage1/proportions.tsv`, `cell_type_profiles.tsv` | Main pipeline's proportions at the chosen k - **no `correlation.tsv`**, since `--refs` wasn't passed here |
| Stage 1 - with refs, k=3 | `xdec.py stage1 --probes ... --refs ... --ref-metadata ... --k 3` | same + `refs.tsv`/`ref_metadata.tsv` | `xdec/stage1_with_refs/{proportions.tsv, cell_type_profiles.tsv, correlation.tsv}` | Adds `correlation.tsv` for option-1 (correlation-based) cell-type labeling at k=3 |
| Stage 1 - with refs, k=8 | same, `--k 8` | same | `xdec/stage1_with_refs_k8/{...}` | Same, at k=8 (the near-tied stability candidate) |
| Stage 2 - main (k=3, no-refs proportions) | `xdec.py stage2 --proportions xdec/stage1/proportions.tsv` | `stage1/proportions.tsv`, bulk counts | `xdec/stage2/{stage2_means.tsv, stage2_stderrs.tsv}` | Per-gene, per-profile expression means/stderrs for the main pipeline's k=3 result |
| Stage 2 - with refs, k=3 | `xdec.py stage2 --proportions xdec/stage1_with_refs/proportions.tsv` | `stage1_with_refs/proportions.tsv`, bulk counts | `xdec/stage2_with_refs_k3/{...}` | Matched Stage 2 output for the `--refs`-labeled k=3 proportions (used for option-2 marker-gene lookup) |
| Stage 2 - with refs, k=8 | `xdec.py stage2 --proportions xdec/stage1_with_refs_k8/proportions.tsv` | `stage1_with_refs_k8/proportions.tsv`, bulk counts | `xdec/stage2_with_refs_k8/{...}` | Same, for k=8 |

Note: `stage1/proportions.tsv` (no refs) and `stage1_with_refs/proportions.tsv` (k=3, with refs) are **not guaranteed to be numerically identical**, even at the same k - passing `--refs` changes the joint quantile-normalization step feeding into Stage 1 (Mode A uses refs for joint QN + post-hoc labeling, not for the factorization itself, but the *normalized bulk matrix* that the factorization runs on does differ slightly). This is why Stage 2 was rerun separately against each proportions file rather than reusing the original `stage2/` output for the marker-gene analysis below.